# Proyecto Titanic: Predicción de Supervivencia y Tarifas (CRISP-ML)

Este cuaderno documenta el análisis y modelado del conjunto de datos del Titanic siguiendo la metodología **CRISP-ML(Q)**. Nuestro objetivo es clasificar la supervivencia de los pasajeros y predecir el costo del pasaje (Fare).

## Fase 1: Comprensión del Negocio y los Datos

### Objetivo del Proyecto
1. **Clasificación (Variable Categórica):** Determinar la probabilidad y clasificar si un pasajero sobrevivió (`Survived`) basado en factores como clase socioeconómica, edad, género y relaciones familiares.
2. **Regresión (Variable Continua):** Predecir el costo del pasaje (`Fare`) pagado por el pasajero a partir de sus atributos.

### Descripción del Dataset
El conjunto de datos contiene información sobre 891 pasajeros del Titanic. Las columnas principales son:
- `PassengerId`: Identificador único.
- `Survived`: 1 si sobrevivió, 0 si no.
- `Pclass`: Clase de boleto (1 = Primera, 2 = Segunda, 3 = Tercera).
- `Name`: Nombre del pasajero.
- `Sex`: Género (male/female).
- `Age`: Edad en años.
- `SibSp`: Número de hermanos / cónyuges a bordo.
- `Parch`: Número de padres / hijos a bordo.
- `Ticket`: Número de boleto.
- `Fare`: Tarifa pagada por el pasajero.
- `Cabin`: Número de cabina.
- `Embarked`: Puerto de embarque (C = Cherbourg, Q = Queenstown, S = Southampton).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, r2_score, mean_absolute_error, mean_squared_error

# Cargar datos
df = pd.read_csv('titanic.csv')
df.head()

In [ ]:
# Información general del dataset
print(df.info())
print("\nValores nulos por columna:")
print(df.isnull().sum())

## Fase 2: Preparación de los Datos

En esta fase limpiaremos los valores nulos e imputaremos valores para las columnas necesarias:
- `Age`: Imputación con la mediana.
- `Embarked`: Imputación con la moda.
- `Fare`: Imputación con la mediana en caso de nulos.
- Codificación de variables categóricas (`Sex` y `Embarked`) a formatos numéricos legibles por los algoritmos.

In [ ]:
# Imputación de nulos
age_median = df['Age'].median()
df['Age'] = df['Age'].fillna(age_median)

embarked_mode = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(embarked_mode)

fare_median = df['Fare'].median()
df['Fare'] = df['Fare'].fillna(fare_median)

# Codificación de categóricas
df['Sex_encoded'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked_encoded'] = df['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

print("Valores nulos después de la preparación:")
print(df[['Age', 'Embarked', 'Fare', 'Sex_encoded', 'Embarked_encoded']].isnull().sum())

## Fase 3: Modelado

### Modelo 1: Regresión Logística (Clasificación de Supervivencia)
Entrenaremos un modelo para clasificar si un pasajero sobrevive o no.

In [ ]:
# Variables predictoras y objetivo
features_clf = ['Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_encoded']
X_clf = df[features_clf]
y_clf = df['Survived']

# División entrenamiento/prueba
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42)

# Logistic Regression
clf_model = LogisticRegression(max_iter=1000)
clf_model.fit(X_train_clf, y_train_clf)
print("Modelo LogisticRegression entrenado con éxito.")

### Modelo 2: Regresión Lineal (Predicción de Tarifa - Variable Continua)
Entrenaremos un modelo lineal para estimar el precio del pasaje (`Fare`) basado en las otras variables (incluyendo `Survived`).

In [ ]:
# Variables predictoras y objetivo para regresión
features_reg = ['Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch', 'Embarked_encoded', 'Survived']
X_reg = df[features_reg]
y_reg = df['Fare']

# División entrenamiento/prueba
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# Linear Regression
reg_model = LinearRegression()
reg_model.fit(X_train_reg, y_train_reg)
print("Modelo LinearRegression entrenado con éxito.")

## Fase 4: Evaluación

Evaluaremos los dos modelos entrenados utilizando métricas estándar.

In [ ]:
# Evaluación de Clasificación (Logistic Regression)
y_pred_clf = clf_model.predict(X_test_clf)
accuracy = accuracy_score(y_test_clf, y_pred_clf)
cm = confusion_matrix(y_test_clf, y_pred_clf)

print("=== Evaluación de Clasificación ===")
print(f"Accuracy: {accuracy:.4f}")
print("\nMatriz de Confusión:")
print(cm)
print("\nReporte de Clasificación:")
print(classification_report(y_test_clf, y_pred_clf))

# Visualizar Matriz de Confusión
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Sobrevive', 'Sobrevive'], yticklabels=['No Sobrevive', 'Sobrevive'])
plt.title('Matriz de Confusión - Supervivencia')
plt.ylabel('Etiqueta Real')
plt.xlabel('Etiqueta Predicha')
plt.show()

In [ ]:
# Evaluación de Regresión (Linear Regression)
y_pred_reg = reg_model.predict(X_test_reg)
r2 = r2_score(y_test_reg, y_pred_reg)
mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)

print("=== Evaluación de Regresión ===")
print(f"R2 Score (Coeficiente de Determinación): {r2:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Mean Squared Error (MSE): {mse:.4f}")

# Gráfico de Predicción vs Valor Real
plt.figure(figsize=(6,4))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.5, color='orange')
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'k--', lw=2)
plt.title('Tarifa Real vs Tarifa Predicha')
plt.xlabel('Tarifa Real ($)')
plt.ylabel('Tarifa Predicha ($)')
plt.show()